#Chapter 3 self attention



#Importing the tokenized stuff from tokenizer file

In [2]:
#OPENONG AND READING fILE
with open("the-verdict.txt","r",encoding="utf-8") as f:
    raw_text=f.read()

    

In [3]:
# Now Byte pair encoding
#using tiktoken

import tiktoken
# version check for tiktoken
from importlib.metadata import version
print("tiktoken version:", version("tiktoken"))

# now actual byte pair encoding using tiktoken
TTokenizer=tiktoken.get_encoding("gpt2")

tiktoken version: 0.13.0


In [4]:

# Dataset and Dataloader-> Dataloader will return x and y in batches, so we can train the model on those batches

# using the pytorch dataset and dataloader

import torch
from torch.utils.data import Dataset, DataLoader


#making a dataset class

class LLMDatasetV1(Dataset):
#text is the whole text, tokenizer is the tiktoken tokenizer, max_length is the context size, stride is how many tokens to move forward for the next input
    def __init__(self,text,tokenizer,max_length,stride):
        self.input_ids=[]
        self.target_ids=[]

        token_ids=TTokenizer.encode(text)

        for i in range(0,len(token_ids)-max_length,stride):
            input_chunk= token_ids[i:i+max_length]
            target_chunks=token_ids[i+1:i+max_length+1]
            self.input_ids.append(torch.tensor(input_chunk))  
            self.target_ids.append(torch.tensor(target_chunks))

    def __len__(self):
        return len(self.input_ids)
    
    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]
    



# Now the dataloader function 

def create_DataloaderV1(text, batch_size=4, max_length=256, stride=128,shuffle=True, drop_last=True, num_workers=0):
    
    dataset= LLMDatasetV1(text,TTokenizer,max_length,stride)
    dataLoader= DataLoader(dataset,batch_size=batch_size,shuffle=shuffle,
                           drop_last=drop_last,num_workers=num_workers)
    return dataLoader




In [5]:
# CALLING BOTH
max_length=4
dataloader= create_DataloaderV1(raw_text,batch_size=8,max_length=4,stride=4,shuffle=False)
data_iter=iter(dataloader)

inputs,targets = next(data_iter)
print("Inputs \n",inputs)
print("Targets \n",targets)

Inputs 
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])
Targets 
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


In [6]:
# Conversion to embeddings -> we are using tiktoken'z gpt2 tokenizer and its vocab is 50257 words
Tvocab_size=50257
output_dim=256
token_embedding_layer=torch.nn.Embedding(Tvocab_size,output_dim)
# now we see the batch previously implemented
print("Inputs \n",inputs)
print("Inputs Shape \n",inputs.shape)

# Now the embeddings
token_embeddings=token_embedding_layer(inputs)
print(token_embeddings.shape)
print(token_embeddings)


Inputs 
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])
Inputs Shape 
 torch.Size([8, 4])
torch.Size([8, 4, 256])
tensor([[[-2.0652e-02, -8.2459e-01, -1.9153e-02,  ..., -7.7140e-01,
           3.8225e-01,  1.5443e-01],
         [ 1.3745e+00,  4.2969e-01,  8.1290e-02,  ..., -9.4652e-01,
          -3.0025e-01, -2.1050e-01],
         [-1.6499e+00,  1.3229e-01,  3.6499e-01,  ...,  1.5084e+00,
           1.3564e+00, -1.4265e+00],
         [ 2.1246e+00,  1.3883e+00,  1.2734e+00,  ..., -6.2199e-02,
           2.6410e-01,  5.8670e-01]],

        [[-1.4240e+00,  1.5625e-01, -8.2256e-01,  ..., -8.2070e-01,
           1.9361e-01,  6.2161e-01],
         [ 9.2703e-01,  1.1349e+00, -1.7446e+00,  ...,  2.6442e-02,
           1.2534e-01, -1.5505e+0

In [7]:
#positional embeddings
# now the positonal embeddings, we need another embedding layer
#max_length is 4
context_length= max_length
pos_embedding_layer=torch.nn.Embedding(context_length,output_dim)
pos_embeddings=pos_embedding_layer(torch.arange(context_length))
print(pos_embeddings.shape)

#add it to token embeddings 
input_embeddings= token_embeddings + pos_embeddings
print(input_embeddings.shape)


torch.Size([4, 256])
torch.Size([8, 4, 256])


# NOW the Actual Chapter 3 --> Self attention

In [8]:
#Self attention

# new text
stext= "Your Journey starts with one step"

# taking example tensor for this
import torch
inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)

)

# we first need to get attention scores for each element with the query
#query is the input sequence/token we want to get the context vector for
#context vector is the enhanced embedded vector for a token, containing information with respect to other tokens

#taking token 2 as query--> journey

input_query=inputs[1]
input_1=inputs[0]

#now we need to take dot product
torch.dot(input_query,input_1)

#but we need to automate for getting this for all inputs relative to the input query

print(inputs.shape[0])
attention_score2=torch.empty(inputs.shape[0])

for i, inp in enumerate(inputs):
    attention_score2[i]=torch.dot(inp,input_query) 

# this prints the attention scores for each input with respect to the query. Higher score = more dependence
print(attention_score2)

# we need to normalize it for easy operation --> we can either use division by sum or softmax function
# addition example
attention_score2_normD=attention_score2/attention_score2.sum()
print(attention_score2_normD)
#these normalized attention scores sum upto 1 now
print("Thier sum is : " ,attention_score2_normD.sum())

# using the python softmax function for best overall result and optimized code
attention_score2_normP=torch.softmax(attention_score2,dim=0)
print("Attention Weights: ",attention_score2_normP)
print("Sum of all weights : ", attention_score2_normP)


#NOW we need to multiple all attention weights with all the input vectors to get 
# the final resultant z enhanced context vector for the query

context_vec_2=torch.zeros(input_query.shape)
for i, inp in enumerate(inputs):
    context_vec_2+=attention_score2_normP[i]*inp
print("Final Context vector for Query T2: "  , context_vec_2)



6
tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])
tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])
Thier sum is :  tensor(1.0000)
Attention Weights:  tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum of all weights :  tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Final Context vector for Query T2:  tensor([0.4419, 0.6515, 0.5683])


# Day 4 (after a break of one day)
# continuing self attention and revision of old code


In [9]:
token_ids=TTokenizer.encode(raw_text)
print(len(token_ids))

# now doing context vectors for all at once

# first the attention scores for all 6 elements as queries with all 6 as the inputs as well
# 6x6 matrix to hold attention scores for all as queries
attention_scores=torch.empty(6,6)

for i, xi in enumerate(inputs):
    for j, xj in enumerate(inputs):
        attention_scores[i,j]=torch.dot(xi,xj)

#print(attention_scores)


#well python magic, we can use matrix multiplication which is faster than the double for loop
#fact, i looked up and visualized, in c++ this code is 3 for loops, which is what this double for loop and matrix
#multiplication does at the backend (pytorch). and for the batch 8x4x256, it is 4 for loops in c++

MMattention_scores= inputs @ inputs.T
#print(MMattention_scores)

# This prints the same tensor, GG

# now we need to normalize it

# we will pass dimension -1 to automatically get the dimension as per the last element

norm_mmattention_scores=torch.softmax(MMattention_scores, dim=-1)
print(norm_mmattention_scores)

# we can verify the normalization by checking sum of all rows
print(norm_mmattention_scores.sum(dim=-1))

#Now we will calculate all the actual context vectors z enhanced for all the token ids

# we can use matrix multiplication again

all_context_vectors=norm_mmattention_scores @ inputs
print(all_context_vectors)


5145
tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])
tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])
tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])


In [16]:
#Now we are adding trainable weights to the self attention
# This enables us to train LLMs to get good vector (steering)

# we need 3 extra matrices, query,value and key-> these are 3 weighted matrices

# using the previous inputs tensor

# first defining 3 variables

# considering again "Journey" as the query

x_2=inputs[1]
dim_in=inputs.shape[1] # input dimension is same as the query
dim_out=2 #output dimension is 2, this is for following the code purposes, actually both input and output dims 
        #are same

# Now we will generalize 3 weight matrices, Query,Key and Value, assign random values and set gradient to off
# in actuallity gradient is on to be able to train, here it is just for understanding (Thanks Andrej for Gradient knowledge)
# we are using torch paramaters to be able to dial up/down
torch.manual_seed(123)
W_query=torch.nn.Parameter(torch.rand(dim_in,dim_out), requires_grad=False)
W_key=torch.nn.Parameter(torch.rand(dim_in,dim_out), requires_grad=False)
W_value=torch.nn.Parameter(torch.rand(dim_in,dim_out), requires_grad=False)


#now we calculate these weights with respect to the query "Journey"

x2query=x_2 @ W_query
x2key=x_2 @ W_key
x2value=x_2 @W_value

print(x2query)
print(x2key)
print(x2value)





tensor([0.4306, 1.4551])
tensor([0.4433, 1.1419])
tensor([0.3951, 1.0037])


# Day 5 (after again break)
# Continuing on from last point


In [28]:
#continuing on 

# we still need to find key and value vectors for all the inputs, even though we are doing this for 
# the 2nd input vector only

keys=inputs @ W_key
values = inputs @ W_value
print("Keys shape: ",  keys.shape)
print("Values shape: ", values.shape)

# now we need to find attention scores, we find em by multiplying query and key vectors
# it is different from simple self attention as that only relied on query and the inputs themselves
# this time we are generating query (from the query) and keys and values for all

# Computing the attention score for Query 2

# first doing it for attention score w22 (query 2 with itself)
# first we find key for the query 2
W_key_2=keys[1]

W_attention_score2 = x2query.dot (W_key_2) # or we can also use x2key as we previously defined

print(W_attention_score2)

# so this is the attention score of query with key 2 (itself)
# we need to generalize it with all inputs keys
#so we use matrix multiplication again

W_attention_scores2=x2query@keys.T
print(W_attention_scores2)

# the output matches
# this is attention score for query 2 with all they keys

# Now we need to calculate attention weights from these scores, and we do that by using a softmax function

# we need dimension of the keys, then we divide the attention scores by the square root of the keys dimension 
# in softmax to get the attention weights
# we need to do this to scale the values, eg in case of 1000s of dimensions , the  large dot products can result 
#in very small gradients, making training hard and stagnant. This step was in original Attention is all 
# you need paper

dim_key=keys.shape[-1]
print(dim_key) #2 here
W_attention_weights2=torch.softmax(W_attention_scores2/dim_key**0.5, dim=-1)
print(W_attention_weights2)

# Now we need to find the context vector
#here it is the weighted sum (multiply each inputs value vectors with its corresponding weighted attenion)
#  and add all up. We can use matrix multiplication again here for ease
W_context_vector=W_attention_weights2@values
print(W_context_vector)

Keys shape:  torch.Size([6, 2])
Values shape:  torch.Size([6, 2])
tensor(1.8524)
tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440])
2
tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820])
tensor([0.3061, 0.8210])


In [33]:
# Now we will make a python class to do all these steps for us in one go

import torch.nn as nn

#class
class self_Attention_v1(nn.Module):
    def __init__(self,dim_in,dim_out):
        #the overrider for nn.Module
        super(). __init__()
        self.w_query=nn.Parameter(torch.rand(dim_in,dim_out))
        self.w_key=nn.Parameter(torch.rand(dim_in,dim_out))
        self.w_value=nn.Parameter(torch.rand(dim_in,dim_out))

    def forward(self,x): # here x is the each input, we will simply pass the whole inputs
            keys=x@self.w_key
            values=x@self.w_value
            queries=x@self.w_query

            attention_scores=queries@keys.T
            attention_weights=torch.softmax(attention_scores/ keys.shape[-1]**0.5,dim=-1)
            context_vecs=attention_weights@values
            return context_vecs



# using the class
torch.manual_seed(123)
self_atts=self_Attention_v1(dim_in,dim_out)

print(self_atts(inputs))

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)


In [ ]:
# we can also use nn.linear instead of parameters. It has more optimized weight intiliazation leading to better
#model training

# implementing the class
class self_Attention_v2(nn.Module):
    def __init__(self,dim_in,dim_out,qkv_bias=False):
        super(). __init__()
        self.w_query=nn.Linear(dim_in,dim_out,bias=qkv_bias)
        self.w_key=nn.Linear(dim_in,dim_out,bias= qkv_bias)
        self.w_value=nn.Linear(dim_in,dim_out,bias =qkv_bias)

    def forward(self,x):
        queries=self.w_query(x)
        keys=self.w_key(x)
        values=self.w_value(x)

        attention_scores=queries@keys.T
        attention_weights=torch.softmax(attention_scores/keys.shape[-1]**0.5, dim=-1)
        context_vecs=attention_weights@values

        return context_vecs


#we can call this class self_Attention_v2

torch.manual_seed(789)
self_Attv2=self_Attention_v2(dim_in,dim_out)
print(self_Attv2(inputs))

# next we need to implement casual attention (hiding future tokens) 

tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)
